# 1. Cleaning Data
Membersihkan data mentah World Bank RTFP (harga pangan bulanan per pasar) menjadi `data_bersih.csv`.

In [3]:
import os
import pandas as pd

raw_path = '../data/raw/IDN_RTFP_mkt_2007_2026-08-24.csv'
output_path = '../data/processed/data_bersih.csv'
missing_path = '../data/processed/laporan_missing.csv'

KOL_PASAR = 'mkt_name'
KOL_PROVINSI = 'adm1_name'
KOL_KABUPATEN = 'adm2_name'
KOL_TANGGAL = 'date'

## Load data

In [4]:
df = pd.read_csv(raw_path)
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={'price_date': KOL_TANGGAL}) if KOL_TANGGAL not in df.columns else df
print(f'Ukuran data awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran data awal: 52,628 baris x 88 kolom


,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,date,year,...,l_sugar,c_sugar,inflation_sugar,trust_sugar,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
0,IDN,Indonesia,SULAWESI UTARA,KOTA KOTAMOBAGU,Pasar 23 Maret,0.74,124.31,gid_74000001243100000,2007-01-01,2007,...,10126.90,10275.85,NaN,8.8,0.68,0.69,0.67,0.68,NaN,9.2
1,IDN,Indonesia,SULAWESI UTARA,KOTA KOTAMOBAGU,Pasar 23 Maret,0.74,124.31,gid_74000001243100000,2007-02-01,2007,...,10157.12,10256.13,NaN,8.8,0.68,0.69,0.67,0.68,NaN,9.2
2,IDN,Indonesia,SULAWESI UTARA,KOTA KOTAMOBAGU,Pasar 23 Maret,0.74,124.31,gid_74000001243100000,2007-03-01,2007,...,10154.80,10355.07,NaN,8.8,0.68,0.69,0.67,0.69,NaN,9.2
3,IDN,Indonesia,SULAWESI UTARA,KOTA KOTAMOBAGU,Pasar 23 Maret,0.74,124.31,gid_74000001243100000,2007-04-01,2007,...,10262.40,10335.94,NaN,8.8,0.69,0.70,0.68,0.70,NaN,9.2
4,IDN,Indonesia,SULAWESI UTARA,KOTA KOTAMOBAGU,Pasar 23 Maret,0.74,124.31,gid_74000001243100000,2007-05-01,2007,...,10002.68,10002.68,NaN,8.8,0.70,0.71,0.69,0.69,NaN,9.2


In [5]:
kolom_wajib = [KOL_PASAR, KOL_TANGGAL]
kolom_hilang = [c for c in kolom_wajib if c not in df.columns]
assert not kolom_hilang, f'Kolom {kolom_hilang} tidak ditemukan, kolom yang ada: {df.columns.tolist()}'

## Hapus baris Market Average

In [6]:
is_average = df[KOL_PASAR].astype(str).str.contains('average', case=False, na=False)
print(f'Baris market average dihapus: {is_average.sum():,}')
print(df.loc[is_average, KOL_PASAR].unique().tolist())
df_clean = df[~is_average].copy()

Baris market average dihapus: 236
['Market Average']


## Hapus kolom yang hanya berisi satu nilai

In [7]:
kolom_konstan = [c for c in df_clean.columns
                 if c not in (KOL_PASAR, KOL_TANGGAL) and df_clean[c].nunique(dropna=True) <= 1]
print('Kolom satu nilai dihapus:', kolom_konstan if kolom_konstan else 'tidak ada')
df_clean = df_clean.drop(columns=kolom_konstan)

Kolom satu nilai dihapus: ['ISO3', 'country', 'currency', 'components', 'start_dense_data', 'last_survey_point', 'data_coverage', 'data_coverage_recent', 'index_confidence_score', 'spatially_interpolated']


## Format tanggal

In [8]:
df_clean[KOL_TANGGAL] = pd.to_datetime(df_clean[KOL_TANGGAL], errors='coerce')
n_tanggal_rusak = df_clean[KOL_TANGGAL].isna().sum()
print(f'Tanggal tidak valid: {n_tanggal_rusak:,}')
df_clean = df_clean.dropna(subset=[KOL_TANGGAL])

Tanggal tidak valid: 0


## Cek duplikat

In [9]:
n_duplikat = df_clean.duplicated().sum()
print(f'Duplikat baris persis (dihapus): {n_duplikat:,}')
df_clean = df_clean.drop_duplicates()

Duplikat baris persis (dihapus): 0


In [10]:
dup_kunci = df_clean[df_clean.duplicated(subset=[KOL_PASAR, KOL_TANGGAL], keep=False)]
print(f'Duplikat pasar dan tanggal dengan nilai berbeda (hanya dilaporkan): {len(dup_kunci):,} baris')
dup_kunci.sort_values([KOL_PASAR, KOL_TANGGAL])[[KOL_PASAR, KOL_TANGGAL]].head(10)

Duplikat pasar dan tanggal dengan nilai berbeda (hanya dilaporkan): 0 baris


,mkt_name,date


## Urutkan data

In [11]:
df_clean = df_clean.sort_values([KOL_PASAR, KOL_TANGGAL]).reset_index(drop=True)

## Missing value (hanya dilaporkan, tidak diisi)

In [12]:
missing = df_clean.isna().sum()
missing = missing[missing > 0].to_frame('jumlah_kosong')
missing['persen'] = (missing['jumlah_kosong'] / len(df_clean) * 100).round(2)
missing = missing.sort_values('persen', ascending=False)
print(f'Kolom yang punya missing value: {len(missing)} dari {df_clean.shape[1]} kolom')
missing.head(15)

Kolom yang punya missing value: 19 dari 78 kolom


,jumlah_kosong,persen
meat_chicken_broiler,41712,79.62
meat_beef,37891,72.32
rice,37694,71.95
meat_chicken,37270,71.14
onions,36935,70.50
eggs,36850,70.34
sugar,36793,70.23
oil,36569,69.80
garlic,36263,69.21
inflation_eggs,2664,5.08


## Simpan hasil
Outlier tidak diubah, sehingga data disimpan apa adanya.

In [13]:
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv(output_path, index=False)
missing.to_csv(missing_path)
print(f'Data bersih disimpan di: {output_path}')
print(f'Laporan missing disimpan di: {missing_path}')

Data bersih disimpan di: ../data/processed/data_bersih.csv
Laporan missing disimpan di: ../data/processed/laporan_missing.csv


## Ringkasan akhir

In [14]:
print(f'Ukuran data: {df.shape[0]:,} x {df.shape[1]}  ->  {df_clean.shape[0]:,} x {df_clean.shape[1]}')
print(f'Jumlah pasar    : {df_clean[KOL_PASAR].nunique()}')
if KOL_PROVINSI in df_clean.columns:
    print(f'Jumlah provinsi : {df_clean[KOL_PROVINSI].nunique()}')
if KOL_KABUPATEN in df_clean.columns:
    print(f'Jumlah kab/kota : {df_clean[KOL_KABUPATEN].nunique()}')
print(f'Periode         : {df_clean[KOL_TANGGAL].min():%b %Y} - {df_clean[KOL_TANGGAL].max():%b %Y}')

Ukuran data: 52,628 x 88  ->  52,392 x 78
Jumlah pasar    : 222
Jumlah provinsi : 34
Jumlah kab/kota : 111
Periode         : Jan 2007 - Aug 2026
